In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset , DataLoader
import pickle

In [2]:
data = datasets.MNIST('./Data',train = True , download = True)

In [3]:
X = data.data
y = data.targets

X_train , X_test , y_train , y_test = train_test_split(X,y,test_size = 0.2,shuffle = True)
X_train =  X_train.reshape(-1,28*28)

X_train = X_train.float()/255

Y_train = y_train.long()

x_test = X_test.float()
x_test = x_test.reshape(-1,28*28)/255

y_test = y_test.long()



In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

device


device(type='cpu')

In [5]:
class Network(nn.Module):
  def __init__(self,size):
    super().__init__()
    self.linear = nn.Sequential(
        nn.Linear(size,100),
        nn.ReLU(),
        nn.Linear(100,75),
        nn.ReLU(),
        nn.Linear(75,10),
    )

  def forward(self,X):
    out = self.linear(X)
    return out

class Loader(Dataset):
  def __init__(self,x,y):
    self.x = x
    self.y = y
  def __len__(self):
    return self.x.size(0)
  def __getitem__(self,item):
    return self.x[item],self.y[item]


In [6]:
Train_loader = Loader(X_train,y_train)
Train_loader = DataLoader(Train_loader,batch_size = 32,shuffle = True)


In [7]:

model = Network(28*28).to(device)
optimizer = optim.Adam(model.parameters(),lr = 0.001)
criteria = nn.CrossEntropyLoss()

In [8]:
epoch = 20
for e in range(epoch):

  total_loss = 0

  for X,y in Train_loader:

    X = X.to(device)
    y = y.to(device)

    y_pred = model(X)

    loss = criteria(y_pred,y)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    total_loss += loss.item()
  print(f"epoch = {e+1} , loss = {total_loss/len(Train_loader)}")



epoch = 1 , loss = 0.3249665700787058
epoch = 2 , loss = 0.13536632493246967
epoch = 3 , loss = 0.09341971611091868
epoch = 4 , loss = 0.07069951741071417
epoch = 5 , loss = 0.055227493868287035
epoch = 6 , loss = 0.04417902001255425
epoch = 7 , loss = 0.036771807217271996
epoch = 8 , loss = 0.029402273153920154
epoch = 9 , loss = 0.025715610406669535
epoch = 10 , loss = 0.02053122246311735
epoch = 11 , loss = 0.021125269151576503
epoch = 12 , loss = 0.018034557662183336
epoch = 13 , loss = 0.013660838272067545
epoch = 14 , loss = 0.01397475453361676
epoch = 15 , loss = 0.010664302761753636
epoch = 16 , loss = 0.014231701599021089
epoch = 17 , loss = 0.01197207687488007
epoch = 18 , loss = 0.010042513176501908
epoch = 19 , loss = 0.011044210240760965
epoch = 20 , loss = 0.01087833736083877


In [ ]:

with torch.no_grad():
  # 1. Get model predictions for all test items
  y_out = model(x_test)

  # 2. Get the index of the max value across the 10 outputs (dim=1)
  predictions = torch.argmax(y_out, dim=1)

  # 3. Count how many predictions match the true values
  count = (predictions == y_test).sum().item()

print("correct --> ", count, " accuracy -->", (count / len(y_test)) * 100, "%")

correct -->  11700  accuracy --> 97.5 %
tensor([[-11.4587, -14.1013,  -6.2844,  ..., -13.3912, -22.5273,   3.5036],
        [-32.8546, -11.6243, -32.7327,  ..., -33.2364, -10.8630,  -4.8521],
        [-36.9216,  16.9209, -16.7971,  ...,  -7.4399,  -8.2018,  -8.3072],
        ...,
        [-19.1020,  -0.2346, -13.5065,  ...,  16.4184, -11.0287,  -1.8542],
        [-23.5133, -11.4289,  15.2347,  ...,  -9.7629, -22.5491,  -8.1476],
        [  3.2341, -16.4815, -27.8782,  ..., -15.0931,  -6.2433,  -6.3343]])


In [10]:
print(torch.argmax(y_out,dim=1))
print(y_test)


tensor([4, 5, 1,  ..., 7, 2, 6])
tensor([4, 5, 1,  ..., 7, 2, 6])


In [11]:
pickle.dump(model,open('modelv2.pkl','wb'))

In [23]:
from torchvision import transforms
from PIL import Image

input = Image.open("to_predict/image.png").convert("L")
transform = transforms.ToTensor()
input = transform(input)
print(input.shape)
input = input.reshape(-1,28*28)

output = model(input)

result = torch.argmax(output,dim = 1)

print(result)

torch.Size([1, 28, 28])
tensor([2])
